# Week 1 — Checkpoint 2: Handling Null Values
### Dataset: `house_sale.csv` (bina.az — real estate sale listings)

https://www.kaggle.com/datasets/sehriyarmemmedli/binaaz-sale-project

**Goal:** Define and apply a separate, reasoned strategy for each of the null values found in Checkpoint 1. Instead of handling all gaps the same way (e.g. dropping everything or filling everything with 0), I investigated **why** each column is missing — random (MCAR) or systematic (MNAR) — and based my decision on that.

## 1. Reloading the dataset

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 150)

df = pd.read_csv('house_sale.csv')
print(df.shape)


(100775, 51)


## 2. Overview of missing values (recap)

From Checkpoint 1, we know 19 columns contain null values, ranging from 0.26% to 99.85%. I pull this list up again here so I can refer to it while investigating each group step by step.

In [2]:
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_percent": null_percent}).sort_values("null_percent", ascending=False)
null_summary[null_summary["null_count"] > 0]


,null_count,null_percent
Binanın növü,100621,99.85
featured,97626,96.88
vip,92179,91.47
Torpaq sahəsi,85204,84.55
hour_y,85144,84.49
İpoteka,67836,67.31
mortgage,67836,67.31
shop_title,28757,28.54
shop_name,28757,28.54
products_label,28145,27.93


## 3. Investigation: MCAR or MNAR?

Before dropping or filling anything, I checked whether each column's missingness pattern is related to **another column** (specifically `Kateqoriya`, the property category). If a column is systematically 100% (or 0%) missing for a particular category, this is not random — the field simply doesn't apply to that category (MNAR, structural missingness). If missingness is spread evenly across all categories, it's more likely random (MCAR).

Below I run this check for three key columns: `Mərtəbə` (floor), `Torpaq sahəsi` (land area), `Otaq sayı` (number of rooms).

In [3]:
print("--- Missing rate of 'Mərtəbə' (floor) by Kateqoriya ---")
print(df.groupby("Kateqoriya")["Mərtəbə"].apply(lambda x: x.isnull().mean().round(3)))


--- Missing rate of 'Mərtəbə' (floor) by Kateqoriya ---
Kateqoriya
Həyət evi/Bağ evi    1.0
Köhnə tikili         0.0
Obyekt               1.0
Ofis                 1.0
Qaraj                1.0
Torpaq               1.0
Yeni tikili          0.0
Name: Mərtəbə, dtype: float64


In [4]:
print("--- Missing rate of 'Torpaq sahəsi' (land area) by Kateqoriya ---")
print(df.groupby("Kateqoriya")["Torpaq sahəsi"].apply(lambda x: x.isnull().mean().round(3)))


--- Missing rate of 'Torpaq sahəsi' (land area) by Kateqoriya ---
Kateqoriya
Həyət evi/Bağ evi    0.0
Köhnə tikili         1.0
Obyekt               1.0
Ofis                 1.0
Qaraj                1.0
Torpaq               1.0
Yeni tikili          1.0
Name: Torpaq sahəsi, dtype: float64


In [5]:
print("--- Missing rate of 'Otaq sayı' (number of rooms) by Kateqoriya ---")
print(df.groupby("Kateqoriya")["Otaq sayı"].apply(lambda x: x.isnull().mean().round(3)))


--- Missing rate of 'Otaq sayı' (number of rooms) by Kateqoriya ---
Kateqoriya
Həyət evi/Bağ evi    0.023
Köhnə tikili         0.000
Obyekt               1.000
Ofis                 0.000
Qaraj                1.000
Torpaq               1.000
Yeni tikili          0.000
Name: Otaq sayı, dtype: float64


**Result:** The pattern is completely clear:
- `Mərtəbə` (floor) is only populated for `Köhnə tikili` and `Yeni tikili` (apartment categories, 0% missing); it's 100% missing for every other category (House/Garden, Object, Office, Garage, Land) — which makes **logical sense**, since "floor number" only applies to apartments. This is a textbook **MNAR (structural missing)** pattern.
- `Torpaq sahəsi` (land area) is the opposite — only populated for `Həyət evi/Bağ evi` (house/garden, 0% missing), 100% missing everywhere else. Again **MNAR** — land area only applies to houses.
- `Otaq sayı` (number of rooms) is nearly fully populated for apartments (Köhnə/Yeni tikili) and Office, and 100% missing for `Obyekt`/`Qaraj`/`Torpaq` (again MNAR — room count doesn't apply to these property types), but for `Həyət evi/Bağ evi` only ~2.3% is missing — this small remainder looks more like genuine **MCAR** (the person listing the property simply left the field blank, with no systematic cause).

This distinction matters: filling structural (MNAR) gaps with "0" or a median would create fabricated data (e.g. "floor = 0" for an office would send a misleading signal). So my strategy differs by group.

## 4. Strategy summary (before applying)

| Column(s) | Missing % | MCAR / MNAR | Decision & reasoning |
|---|---|---|---|
| `Binanın növü` (building type) | 99.85% | — | **Drop the column.** Almost entirely empty, provides no usable signal, and its information largely overlaps with the `Kateqoriya` column. |
| `featured` | 96.88% | MNAR | Only one value (`featured`) ever appears — the site only populates this field when the listing is actually featured. Convert to a boolean flag: `featured_flag` (present = True, missing = False). |
| `vip` | 91.47% | MNAR | Same logic — create a `vip_flag` boolean column. |
| `mortgage` / `İpoteka` | 67.31% (both) | MNAR | Verified: these two columns are missing on the **exact same rows** (a duplicate pair left over from an `_x`/`_y` merge). Merged into a single `mortgage_flag` boolean and dropped the duplicate. |
| `Torpaq sahəsi` (land area) | 84.55% | **MNAR (structural)** | Only applies to the `Həyət evi/Bağ evi` category (see check above). I do **not** fill the missing values — filling with 0 would falsely imply "no land," which is wrong (it simply doesn't apply to this property type). Left as NaN, to be interpreted alongside `Kateqoriya` during analysis. |
| `hour_y` | 84.49% | Unclear / MCAR-like | `day_y` is 100% populated but `hour_y` is not — no relationship found with category. The second scrape source appears to have only partially captured the time value. Given its low analytical value, **I dropped the column**. |
| `shop_name` / `shop_title` / `owner_name` / `owner_title` | 28.54% / 0.65% | MNAR + small MCAR | Verified: whenever `shop_name` is filled, `owner_name` is always empty, and vice versa (a listing is posted either by an agency or an individual). Derived a `seller_type` column (Agency / Individual / Unknown) from this, and filled the remaining text fields with `"N/A"`. 657 rows have both empty — marked as `"Unknown"` (MCAR, ~0.65%, negligible). |
| `products_label` | 27.93% | MNAR | Empty when no special label applies → filled with `"Regular listing"`. |
| `unit_price` / `Mərtəbə` | 24.59% (both) | MNAR (structural, same pattern as `Mərtəbə`) | `Mərtəbə`: left as NaN (see above). `unit_price`: instead of filling, **I recalculated it myself** — the `Sahə` (area) column has 0% missing, so I can compute an exact value for every row using `unit_price_calculated = price / Sahə` — more reliable than any imputation method. |
| `bill_of_sale` / `Çıxarış` | 21.05% / different % | Mixed | These two columns are not missing on identical rows (checked), so I didn't treat them as pure duplicates — instead created a single `bill_of_sale_flag` boolean based on whether either field indicates a sale document exists. |
| `repair` / `Təmir` | 19.02% / 5.95% | Mixed (redundant + small MCAR) | `repair` only ever has one value (`Renovated`) and is a subset of `Təmir`, which has two values (`yes`/`no`) and is more complete. Dropped `repair`, and filled `Təmir`'s remaining 5.95% gap with `"Unknown"` (assuming "no" would be a false assumption, since the real status is unknown). |
| `Otaq sayı` (number of rooms) | 9.34% | Mixed (mostly MNAR + small MCAR) | Left the structural portion (Object/Garage/Land) as NaN. Only filled the remaining gaps within relevant categories (apartments, office, house) using **that category's median** — median instead of mean, to avoid outliers (e.g. a 9-room villa) skewing the fill value. |
| `owner_name` / `owner_title` | 0.65% | MCAR | Folded into the `seller_type` logic above. |
| `description` | 0.26% | MCAR | A text field — filled with an empty string (`""`), since it won't be used for numeric calculations. |


## 5. Applying the strategy

To keep a clean before/after comparison, I keep the original `df` untouched and apply all changes on a copy, `df_clean`.

In [6]:
df_clean = df.copy()

# --- Binanın növü: almost entirely empty, not a useful column ---
df_clean = df_clean.drop(columns=["Binanın növü"])
print("'Binanın növü' dropped.")


'Binanın növü' dropped.


In [7]:
# --- featured / vip: convert to boolean flags (missing = False) ---
df_clean["featured_flag"] = df_clean["featured"].notnull()
df_clean["vip_flag"] = df_clean["vip"].notnull()
df_clean = df_clean.drop(columns=["featured", "vip"])
print(df_clean[["featured_flag", "vip_flag"]].mean().round(3))


featured_flag    0.031
vip_flag         0.085
dtype: float64


In [8]:
# --- mortgage / İpoteka: duplicate pair -> single flag ---
df_clean["mortgage_flag"] = df_clean["mortgage"].notnull()
df_clean = df_clean.drop(columns=["mortgage", "İpoteka"])
print(df_clean["mortgage_flag"].mean().round(3))


0.327


In [9]:
# --- bill_of_sale / Çıxarış: combined flag ---
df_clean["bill_of_sale_flag"] = df_clean["bill_of_sale"].notnull() | df_clean["Çıxarış"].notnull()
df_clean = df_clean.drop(columns=["bill_of_sale", "Çıxarış"])
print(df_clean["bill_of_sale_flag"].mean().round(3))


1.0


In [10]:
# --- repair / Təmir: keep Təmir as the primary field, fill its remaining gaps ---
df_clean["Təmir"] = df_clean["Təmir"].fillna("Unknown")
df_clean = df_clean.drop(columns=["repair"])
print(df_clean["Təmir"].value_counts())


Təmir
var        81612
yoxdur     13162
Unknown     6001
Name: count, dtype: int64


In [11]:
# --- Torpaq sahəsi: structural missingness, left as-is (NO imputation) ---
print("Rows intentionally left null for Torpaq sahəsi:", df_clean["Torpaq sahəsi"].isnull().sum())


Rows intentionally left null for Torpaq sahəsi: 85204


In [12]:
# --- hour_y: low analytical value, dropped ---
df_clean = df_clean.drop(columns=["hour_y"])
print("'hour_y' dropped.")


'hour_y' dropped.


In [13]:
# --- shop_name/owner_name group: derive a seller_type column ---
def seller_type(row):
    if pd.notnull(row["shop_name"]):
        return "Agency"
    elif pd.notnull(row["owner_name"]):
        return "Individual"
    else:
        return "Unknown"

df_clean["seller_type"] = df_clean.apply(seller_type, axis=1)

for col in ["shop_name", "shop_title", "owner_name", "owner_title"]:
    df_clean[col] = df_clean[col].fillna("N/A")

print(df_clean["seller_type"].value_counts())


seller_type
Agency        72018
Individual    28100
Unknown         657
Name: count, dtype: int64


In [14]:
# --- products_label: 'Regular listing' when no special tag applies ---
df_clean["products_label"] = df_clean["products_label"].fillna("Regular listing")
print(df_clean["products_label"].value_counts())


products_label
Agentlik           71981
Regular listing    28145
Kompleks             649
Name: count, dtype: int64


In [15]:
# --- Mərtəbə: structural missingness, left as-is ---
print("Rows intentionally left null for Mərtəbə:", df_clean["Mərtəbə"].isnull().sum())


Rows intentionally left null for Mərtəbə: 24779


In [16]:
# --- unit_price: recalculated from the Sahə column (0% missing) ---
# Sahə comes in two units, 'm²' and 'sot' (for land) -> convert everything to m² (1 sot = 100 m²)
def parse_area(val):
    if pd.isnull(val):
        return np.nan
    val = val.strip()
    if "sot" in val:
        return float(val.replace("sot", "").strip().replace(",", ".")) * 100
    return float(val.replace("m²", "").strip().replace(" ", ""))

df_clean["Sahə_numeric"] = df_clean["Sahə"].apply(parse_area)
df_clean["unit_price_calculated"] = (df_clean["price"] / df_clean["Sahə_numeric"]).round(2)
df_clean = df_clean.drop(columns=["unit_price"])

print("Missing rows in unit_price_calculated:", df_clean["unit_price_calculated"].isnull().sum())
df_clean["unit_price_calculated"].describe()


Missing rows in unit_price_calculated: 0


,unit_price_calculated
count,1.007750e+05
mean,2.321759e+03
std,1.042945e+04
min,0.000000e+00
25%,1.675000e+03
50%,2.214290e+03
75%,2.730500e+03
max,3.000000e+06


In [17]:
# --- Otaq sayı: median-fill only within applicable categories ---
applicable_categories = ["Köhnə tikili", "Yeni tikili", "Ofis", "Həyət evi/Bağ evi"]
mask = df_clean["Kateqoriya"].isin(applicable_categories)
medians = df_clean.loc[mask].groupby("Kateqoriya")["Otaq sayı"].median()
print("Median room count by category:")
print(medians)

for cat, med in medians.items():
    idx = df_clean[(df_clean["Kateqoriya"] == cat) & (df_clean["Otaq sayı"].isnull())].index
    df_clean.loc[idx, "Otaq sayı"] = med

print("\nRemaining missing rows for 'Otaq sayı' (structural only, Object/Garage/Land):", df_clean["Otaq sayı"].isnull().sum())


Median room count by category:
Kateqoriya
Həyət evi/Bağ evi    4.0
Köhnə tikili         3.0
Ofis                 4.0
Yeni tikili          3.0
Name: Otaq sayı, dtype: float64

Remaining missing rows for 'Otaq sayı' (structural only, Object/Garage/Land): 9054


In [18]:
# --- description: text field, fill with empty string ---
df_clean["description"] = df_clean["description"].fillna("")
print("Remaining missing rows for 'description':", df_clean["description"].isnull().sum())


Remaining missing rows for 'description': 0


## 6. Final check

After all steps, I re-check the missingness status. Aside from the two columns I intentionally left untouched (`Torpaq sahəsi`, `Mərtəbə`), almost every gap has now either been filled, converted into a flag, or the column dropped.

In [19]:
final_nulls = df_clean.isnull().sum()
final_nulls[final_nulls > 0]


,0
Mərtəbə,24779
Otaq sayı,9054
Torpaq sahəsi,85204


In [20]:
print("Original column count:", df.shape[1])
print("New column count:", df_clean.shape[1])
print("Row count unchanged:", df.shape[0] == df_clean.shape[0])


Original column count: 51
New column count: 48
Row count unchanged: True


## Checkpoint 2 — Summary

- Instead of "blindly" filling every empty column, I first checked whether each gap was **structural (MNAR)** or **random (MCAR)** by cross-referencing against `Kateqoriya`. Result: most of the missingness in `Mərtəbə`, `Torpaq sahəsi`, and `Otaq sayı` is **structural**, tied to property type (e.g. "floor" is meaningless for an office or a plot of land) — filling these with a fabricated value (0/median) would send a misleading signal, so I intentionally left them as NaN.
- "Single-value" columns such as `vip`, `featured`, `mortgage`, `bill_of_sale` were converted into boolean flags — this both eliminates the missingness and produces a more model-friendly format.
- Confirmed that the `mortgage`/`İpoteka` pair is missing on **exactly the same rows** and merged the duplicate — this confirms the `_x`/`_y` merge suspicion noted back in Checkpoint 1.
- Instead of imputing `unit_price`, I **recalculated it myself** from the fully populated `Sahə` and `price` columns — more reliable than any imputation method.
- Traditional fill methods (median / "Unknown") were only used for small, genuinely unexplained gaps (part of `Otaq sayı`, `Təmir`, `owner_name`).
- Row count **stayed unchanged** (no rows were dropped) — only column-level fixes were applied, since no column's missing rate justified discarding rows wholesale.
